## Téléchargement automatique de Fer2013 si absent du dossier courant

In [1]:
# ============================================================
# Téléchargement automatique de FER2013 si absent du dossier courant
# ============================================================

from pathlib import Path
import urllib.request
import zipfile
import shutil

DATA_FILE = Path.cwd() / "fer2013.csv"

if DATA_FILE.exists():
    print(f"✓ Dataset déjà présent : {DATA_FILE}")
else:
    print("FER2013 absent du répertoire courant : téléchargement en cours...")

    # Miroir public du fichier FER2013 au format CSV compressé
    DATASET_URL = (
        "https://huggingface.co/datasets/chitradrishti/fer2013/"
        "resolve/main/fer2013.csv.zip?download=true"
    )
    ZIP_FILE = Path.cwd() / "fer2013.csv.zip"

    try:
        urllib.request.urlretrieve(DATASET_URL, ZIP_FILE)

        with zipfile.ZipFile(ZIP_FILE, "r") as z:
            csv_files = [
                name for name in z.namelist()
                if Path(name).name.lower() == "fer2013.csv"
            ]

            if not csv_files:
                raise FileNotFoundError(
                    "L'archive téléchargée ne contient pas de fichier fer2013.csv."
                )

            # Extrait uniquement le CSV attendu dans le répertoire courant
            with z.open(csv_files[0]) as src, open(DATA_FILE, "wb") as dst:
                shutil.copyfileobj(src, dst)

        print(f"✓ Dataset téléchargé et extrait : {DATA_FILE}")

    finally:
        # Nettoyage de l'archive, même en cas d'erreur après le téléchargement
        if ZIP_FILE.exists():
            ZIP_FILE.unlink()

print(f"Taille : {DATA_FILE.stat().st_size / (1024**2):.1f} Mo")

FER2013 absent du répertoire courant : téléchargement en cours...
✓ Dataset téléchargé et extrait : h:\Downloads\fer2013.csv
Taille : 287.1 Mo


# Projet Deep Learning — Reconnaissance d’expressions faciales avec FER2013

**Dataset choisi : FER2013**  
**Objectif :** construire, entraîner, évaluer et améliorer un classifieur multiclasse d’expressions faciales, puis préparer les extensions éventuelles.

> Ce notebook est une **template structurée** : les cellules marquées `TODO` sont à compléter avec vos résultats, justifications et analyses.
>
> Il suppose en priorité la version CSV classique de FER2013 avec les colonnes `emotion`, `pixels` et éventuellement `Usage`.  
> Si votre version est organisée en dossiers d’images, adaptez uniquement la partie chargement.

### Plan
1. Recherche, compréhension et préparation des données
2. Modèle de référence (réseau dense)
3. Construction d’un CNN avec Keras
4. Entraînement
5. Évaluation et analyse des erreurs
6. Expérimentations et amélioration — **au moins 3 expériences**
7. Enrichissement avec les nouvelles notions du cours
8. Extension optionnelle : détection de plusieurs visages
9. Extension optionnelle : vidéo

## 0. Installation, imports et reproductibilité

**À expliquer dans le rendu :**
- pourquoi fixer une graine aléatoire ;
- versions principales utilisées ;
- environnement d’exécution (Google Colab / GPU si disponible).

In [ ]:
# Si nécessaire dans Colab :
# !pip install -q tensorflow scikit-learn seaborn pandas matplotlib

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow :", tf.__version__)
print("GPU(s) :", tf.config.list_physical_devices("GPU"))

## 1. Recherche, compréhension et préparation des données

### 1.1 Fiche descriptive du dataset

**TODO — compléter avec des sources vérifiées :**

- **Nom :** FER2013
- **Source / lien :** https://www.kaggle.com/datasets/msambare/fer2013
- **Auteurs / organisme :** Manas Sambare
- **Licence / conditions d’utilisation :** Database Contents License (DbCL) v1.0
- **Nombre total d’images :** 32298
- **Nombre de classes :** 7
- **Classes :** 0=Angry, 1=Disgust, 2=Fear, 3=Happy, 4=Sad, 5=Surprise, 6=Neutral
- **Nombre d’images par classe :** 
- **Dimensions des images :** 48 x 48
- **Niveaux de gris ou couleur :** 0-255 gray
- **Format de stockage utilisé dans votre copie :** ??
- **Déséquilibre des classes :** oui / non, justification.

### Hypothèse de travail de cette template

Pour la version CSV courante :
- une ligne = une image ;
- `emotion` = identifiant numérique de la classe ;
- `pixels` = pixels aplatis sous forme de chaîne ;
- `Usage` = découpage fourni (`Training`, `PublicTest`, `PrivateTest`) lorsqu’il existe.

**Vérifiez votre fichier avant d’utiliser ces hypothèses.**

### 1.2 Configuration des chemins

In [ ]:
# ===== À MODIFIER =====
DATA_PATH = "./fer2013.csv"   # téléchargé automatiquement dans le répertoire courant

# Mapping classiquement associé à FER2013.
# Vérifiez qu'il correspond bien à votre copie du dataset.
CLASS_NAMES = {
    0: "Angry",
    1: "Disgust",
    2: "Fear",
    3: "Happy",
    4: "Sad",
    5: "Surprise",
    6: "Neutral",
}

IMG_H = 48
IMG_W = 48
N_CLASSES = 7

### 1.3 Chargement et vérifications initiales

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape du tableau :", df.shape)
display(df.head())

print("\nColonnes :", list(df.columns))
print("\nValeurs manquantes :")
display(df.isna().sum())

In [ ]:
# Contrôles minimaux
required_cols = {"emotion", "pixels"}
missing = required_cols - set(df.columns)
assert not missing, f"Colonnes manquantes : {missing}"

print(df["emotion"].value_counts().sort_index())

### 1.4 Distribution des classes et déséquilibre

**TODO — analyse :**
- Quelles classes sont les plus représentées ?
- Les moins représentées ?
- Le déséquilibre est-il susceptible d’affecter l’accuracy globale ?

In [ ]:
class_counts = df["emotion"].value_counts().sort_index()
class_labels = [CLASS_NAMES.get(i, str(i)) for i in class_counts.index]

plt.figure(figsize=(9, 4))
plt.bar(class_labels, class_counts.values)
plt.xticks(rotation=30)
plt.ylabel("Nombre d'images")
plt.title("Distribution des classes — FER2013")
plt.show()

display(pd.DataFrame({
    "class_id": class_counts.index,
    "class_name": class_labels,
    "count": class_counts.values,
    "percent": 100 * class_counts.values / class_counts.sum()
}))

### 1.5 Conversion des pixels en tenseurs d’images

In [ ]:
def parse_pixels(pixel_string, img_h=48, img_w=48):
    arr = np.fromstring(pixel_string, dtype=np.float32, sep=" ")
    if arr.size != img_h * img_w:
        raise ValueError(f"Taille inattendue : {arr.size} pixels")
    return arr.reshape(img_h, img_w, 1)

X_all = np.stack(df["pixels"].map(parse_pixels).to_numpy())
y_all = df["emotion"].astype("int32").to_numpy()

print("X_all :", X_all.shape, X_all.dtype)
print("y_all :", y_all.shape, y_all.dtype)
print("Pixels min/max avant normalisation :", X_all.min(), X_all.max())

### 1.6 Afficher plusieurs exemples de chaque catégorie

In [ ]:
def show_examples_per_class(X, y, class_names, n_per_class=5):
    classes = sorted(np.unique(y))
    fig, axes = plt.subplots(len(classes), n_per_class, figsize=(2*n_per_class, 2*len(classes)))

    for row, cls in enumerate(classes):
        idx = np.where(y == cls)[0][:n_per_class]
        for col in range(n_per_class):
            ax = axes[row, col]
            if col < len(idx):
                ax.imshow(X[idx[col]].squeeze(), cmap="gray")
                ax.set_title(class_names.get(int(cls), str(cls)))
            ax.axis("off")
    plt.tight_layout()
    plt.show()

show_examples_per_class(X_all, y_all, CLASS_NAMES, n_per_class=5)

### 1.7 Normalisation des pixels

On transforme les intensités de pixels en valeurs comprises entre 0 et 1.

**TODO — expliquer :**
- pourquoi la normalisation facilite l’optimisation ;
- pourquoi il faut appliquer le même prétraitement au train, à la validation, au test et plus tard aux visages détectés.

In [ ]:
X_all = X_all / 255.0
print("Pixels min/max après normalisation :", X_all.min(), X_all.max())

### 1.8 Constitution de train / validation / test

Le sujet impose de réserver le **jeu de test à l’évaluation finale**.

Deux cas sont prévus ci-dessous :

1. `Usage` existe : on respecte le découpage fourni ;
2. sinon : on crée un split stratifié.

In [ ]:
if "Usage" in df.columns:
    usage = df["Usage"].astype(str).to_numpy()

    train_mask = usage == "Training"
    val_mask   = usage == "PublicTest"
    test_mask  = usage == "PrivateTest"

    X_train, y_train = X_all[train_mask], y_all[train_mask]
    X_val,   y_val   = X_all[val_mask],   y_all[val_mask]
    X_test,  y_test  = X_all[test_mask],  y_all[test_mask]

    print("Split basé sur la colonne Usage.")
else:
    X_temp, X_test, y_temp, y_test = train_test_split(
        X_all, y_all,
        test_size=0.15,
        random_state=SEED,
        stratify=y_all
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=0.15 / 0.85,
        random_state=SEED,
        stratify=y_temp
    )

    print("Split stratifié créé manuellement.")

print("Train :", X_train.shape, y_train.shape)
print("Val   :", X_val.shape, y_val.shape)
print("Test  :", X_test.shape, y_test.shape)

### 1.9 Vérification de la répartition des classes après split

In [ ]:
def split_distribution(y, name):
    vc = pd.Series(y).value_counts().sort_index()
    out = pd.DataFrame({
        "class_id": vc.index,
        "class_name": [CLASS_NAMES.get(int(i), str(i)) for i in vc.index],
        "count": vc.values,
        "percent": 100 * vc.values / vc.sum()
    })
    out["split"] = name
    return out

display(pd.concat([
    split_distribution(y_train, "train"),
    split_distribution(y_val, "validation"),
    split_distribution(y_test, "test"),
], ignore_index=True))

## 2. Modèle de référence — réseau dense

Architecture de référence :

`Image → Flatten → Dense + activation → Dense → Softmax`

L’objectif est de disposer d’un **baseline** simple, pas d’obtenir la meilleure performance.

**TODO — être capable d’expliquer :**
- Flatten ;
- poids et biais ;
- propagation avant ;
- fonctions d’activation ;
- fonction de perte ;
- rétropropagation ;
- descente de gradient ;
- sortie Softmax multiclasse.

In [ ]:
def build_baseline(input_shape=(48, 48, 1), n_classes=7):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dense(n_classes, activation="softmax")
    ], name="baseline_dense")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

baseline = build_baseline((IMG_H, IMG_W, 1), N_CLASSES)
baseline.summary()

### 2.1 Entraînement du baseline

In [ ]:
baseline_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    )
]

history_baseline = baseline.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=128,
    callbacks=baseline_callbacks,
    verbose=1
)

In [ ]:
baseline_val_loss, baseline_val_acc = baseline.evaluate(X_val, y_val, verbose=0)
print(f"Baseline — val_loss={baseline_val_loss:.4f}, val_accuracy={baseline_val_acc:.4f}")

## 3. Construction d’un CNN avec Keras

Le CNN doit contenir plusieurs blocs de convolution et de pooling.

**TODO — justifier vos choix :**
- nombre de blocs convolutionnels ;
- nombre de filtres ;
- taille des kernels ;
- padding ;
- pooling ;
- activation ReLU ;
- Flatten ou GlobalAveragePooling ;
- Dense ;
- couche finale Softmax.

In [ ]:
def build_cnn(
    input_shape=(48, 48, 1),
    n_classes=7,
    dropout_rate=0.30,
    learning_rate=1e-3
):
    model = keras.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(dropout_rate),
        layers.Dense(n_classes, activation="softmax"),
    ], name="cnn_fer2013")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

cnn = build_cnn((IMG_H, IMG_W, 1), N_CLASSES)
cnn.summary()

### 3.1 Tableau couche par couche : dimensions et nombre de paramètres

Le sujet demande de suivre l’évolution des dimensions dans le réseau et le nombre de paramètres de chaque couche.

In [ ]:
def model_layers_table(model):
    rows = []
    for layer in model.layers:
        try:
            output_shape = tuple(layer.output.shape)
        except Exception:
            output_shape = "?"
        rows.append({
            "Layer": layer.name,
            "Type": layer.__class__.__name__,
            "Output shape": output_shape,
            "Params": layer.count_params()
        })
    return pd.DataFrame(rows)

display(model_layers_table(cnn))
print("Nombre total de paramètres :", cnn.count_params())

**TODO — commentaire :**
- Comment évoluent hauteur / largeur après chaque pooling ?
- Pourquoi le nombre de feature maps augmente-t-il ?
- Quelle partie du réseau concentre le plus de paramètres ?

## 4. Entraînement du CNN

**Choix à justifier :**
- fonction de perte ;
- optimiseur ;
- learning rate ;
- batch size ;
- nombre maximal d’époques ;
- métriques ;
- éventuel Early Stopping.

In [ ]:
EPOCHS = 40
BATCH_SIZE = 128

cnn_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        "best_cnn.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

history_cnn = cnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cnn_callbacks,
    verbose=1
)

### 4.1 Courbes d’apprentissage

In [ ]:
def plot_history(history, title="Training history"):
    hist = pd.DataFrame(history.history)

    plt.figure(figsize=(7, 4))
    plt.plot(hist.index + 1, hist["loss"], label="train loss")
    plt.plot(hist.index + 1, hist["val_loss"], label="val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title + " — Loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(hist.index + 1, hist["accuracy"], label="train accuracy")
    plt.plot(hist.index + 1, hist["val_accuracy"], label="val accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title + " — Accuracy")
    plt.legend()
    plt.show()

plot_history(history_cnn, "CNN FER2013")

### 4.2 Analyse du surapprentissage

**TODO — interpréter les courbes :**
- À partir de quelle époque la validation cesse-t-elle de s’améliorer ?
- Y a-t-il un écart croissant entre train et validation ?
- Le modèle sous-apprend-il ou surapprend-il ?
- Quel effet a l’Early Stopping ?

## 5. Évaluation et analyse des erreurs

L’accuracy seule ne suffit pas. Cette partie doit contenir :
- matrice de confusion ;
- métriques par classe ;
- expressions bien reconnues ;
- expressions souvent confondues ;
- exemples de bonnes et mauvaises prédictions.

### 5.1 Évaluation finale sur le jeu de test

⚠️ **À exécuter uniquement lorsque le modèle final est choisi.**

In [ ]:
# Recharge le meilleur modèle sauvegardé si nécessaire :
# final_model = keras.models.load_model("best_cnn.keras")

final_model = cnn

test_loss, test_acc = final_model.evaluate(X_test, y_test, verbose=0)
print(f"TEST — loss={test_loss:.4f}, accuracy={test_acc:.4f}")

In [ ]:
y_proba = final_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_proba, axis=1)

print(classification_report(
    y_test,
    y_pred,
    target_names=[CLASS_NAMES[i] for i in range(N_CLASSES)],
    digits=4
))

### 5.2 Matrice de confusion

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap="Blues")
plt.title("Matrice de confusion — Test")
plt.xlabel("Classe prédite")
plt.ylabel("Vraie classe")
plt.xticks(range(N_CLASSES), [CLASS_NAMES[i] for i in range(N_CLASSES)], rotation=45)
plt.yticks(range(N_CLASSES), [CLASS_NAMES[i] for i in range(N_CLASSES)])

for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()

**TODO — analyse critique :**
- Quelles classes ont les meilleurs rappels ?
- Quelles paires de classes sont les plus souvent confondues ?
- Le déséquilibre de FER2013 semble-t-il jouer un rôle ?
- Quelles limites visuelles du dataset peuvent expliquer certaines erreurs ?

### 5.3 Afficher des prédictions : vraie classe + classe prédite + probabilité

In [ ]:
def show_predictions(X, y_true, y_pred, y_proba, n=12, only_errors=False):
    if only_errors:
        indices = np.where(y_true != y_pred)[0]
    else:
        indices = np.arange(len(y_true))

    if len(indices) == 0:
        print("Aucun exemple disponible.")
        return

    rng = np.random.default_rng(SEED)
    chosen = rng.choice(indices, size=min(n, len(indices)), replace=False)

    cols = 4
    rows = int(np.ceil(len(chosen) / cols))
    plt.figure(figsize=(14, 3.5 * rows))

    for k, idx in enumerate(chosen, start=1):
        plt.subplot(rows, cols, k)
        plt.imshow(X[idx].squeeze(), cmap="gray")
        true_name = CLASS_NAMES[int(y_true[idx])]
        pred_name = CLASS_NAMES[int(y_pred[idx])]
        confidence = float(y_proba[idx, y_pred[idx]])
        plt.title(f"Vrai: {true_name}\nPrédit: {pred_name} ({confidence:.2f})")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_predictions(X_test, y_test, y_pred, y_proba, n=12, only_errors=False)

### 5.4 Analyse ciblée des erreurs

In [ ]:
show_predictions(X_test, y_test, y_pred, y_proba, n=16, only_errors=True)

**TODO — pour plusieurs erreurs, expliquer qualitativement :**
- ambiguïté de l’expression ;
- occlusion / pose / éclairage ;
- visage peu contrasté ;
- ressemblance entre classes ;
- étiquette potentiellement discutable ;
- confiance du modèle élevée ou faible.

## 6. Expérimentations et amélioration — minimum 3 expériences

Règle méthodologique : modifier **un nombre limité de paramètres à la fois** pour pouvoir attribuer l’effet observé.

### Tableau de suivi

| Expérience | Modification | Val accuracy | Val loss | Observation |
|---|---|---:|---:|---|
| Baseline dense | Réseau dense | TODO | TODO | TODO |
| CNN de référence | 3 blocs conv | TODO | TODO | TODO |
| Expérience 1 | TODO | TODO | TODO | TODO |
| Expérience 2 | TODO | TODO | TODO | TODO |
| Expérience 3 | TODO | TODO | TODO | TODO |

### 6.1 Infrastructure d’expérimentation

Cette fonction permet de centraliser les résultats de validation.

In [ ]:
experiment_results = []

def record_experiment(name, model, history):
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    experiment_results.append({
        "experiment": name,
        "best_val_accuracy_history": max(history.history["val_accuracy"]),
        "best_val_loss_history": min(history.history["val_loss"]),
        "final_eval_val_accuracy": val_acc,
        "final_eval_val_loss": val_loss,
        "n_params": model.count_params(),
    })
    return pd.DataFrame(experiment_results)

# Exemple :
# record_experiment("CNN reference", cnn, history_cnn)

### 6.2 Expérience 1 — Data Augmentation

**Question étudiée :** la création de variantes réalistes des images améliore-t-elle la généralisation ?

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.08),
], name="data_augmentation")

def build_cnn_with_augmentation():
    inputs = keras.Input(shape=(IMG_H, IMG_W, 1))
    x = data_augmentation(inputs)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(N_CLASSES, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="cnn_aug")
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# TODO : décommenter pour lancer
# exp1_model = build_cnn_with_augmentation()
# exp1_history = exp1_model.fit(
#     X_train, y_train,
#     validation_data=(X_val, y_val),
#     epochs=40,
#     batch_size=128,
#     callbacks=[
#         callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
#     ]
# )
# display(record_experiment("Exp1 - Data augmentation", exp1_model, exp1_history))

### 6.3 Expérience 2 — Régularisation / Dropout

**Question étudiée :** augmenter le Dropout réduit-il le surapprentissage ?

In [ ]:
# TODO : comparer par exemple dropout_rate=0.30 puis 0.50
# exp2_model = build_cnn(
#     input_shape=(IMG_H, IMG_W, 1),
#     n_classes=N_CLASSES,
#     dropout_rate=0.50,
#     learning_rate=1e-3
# )
#
# exp2_history = exp2_model.fit(
#     X_train, y_train,
#     validation_data=(X_val, y_val),
#     epochs=40,
#     batch_size=128,
#     callbacks=[
#         callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
#     ]
# )
#
# display(record_experiment("Exp2 - Dropout 0.50", exp2_model, exp2_history))

### 6.4 Expérience 3 — Learning rate / optimiseur

**Question étudiée :** un learning rate plus faible stabilise-t-il l’entraînement ?

In [ ]:
# TODO : exemple avec Adam(3e-4)
# exp3_model = build_cnn(
#     input_shape=(IMG_H, IMG_W, 1),
#     n_classes=N_CLASSES,
#     dropout_rate=0.30,
#     learning_rate=3e-4
# )
#
# exp3_history = exp3_model.fit(
#     X_train, y_train,
#     validation_data=(X_val, y_val),
#     epochs=40,
#     batch_size=128,
#     callbacks=[
#         callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
#     ]
# )
#
# display(record_experiment("Exp3 - LR 3e-4", exp3_model, exp3_history))

### 6.5 Tableau comparatif final des expériences

In [ ]:
# Après avoir lancé les expériences :
# results_df = pd.DataFrame(experiment_results).sort_values(
#     "final_eval_val_accuracy",
#     ascending=False
# )
# display(results_df)

### 6.6 Choix du modèle final

**TODO — justification :**
- quelle expérience donne le meilleur compromis validation / généralisation ?
- le gain est-il significatif ?
- le modèle est-il plus complexe ?
- observe-t-on moins de surapprentissage ?
- pourquoi retenir ce modèle plutôt qu’un autre ?

> Le choix final doit être fait à partir de la **validation**, pas du test.

## 7. Enrichissement avec les nouvelles notions du cours

Exemples possibles :
- Data Augmentation plus poussée ;
- Batch Normalization ;
- régularisation L2 ;
- Transfer Learning ;
- modèle pré-entraîné ;
- détection de visages ;
- YOLO.

### Proposition d’enrichissement : Transfer Learning

FER2013 étant en 48×48 niveaux de gris, un backbone ImageNet nécessite généralement :
1. conversion grayscale → RGB ;
2. redimensionnement vers une taille compatible ;
3. prétraitement adapté au backbone ;
4. éventuellement gel puis fine-tuning.

**Cette cellule est volontairement laissée comme extension, car elle change fortement le pipeline.**

In [ ]:
# Exemple de squelette — NE PAS lancer sans adapter soigneusement le pipeline.

# TARGET_SIZE = (96, 96)
#
# def preprocess_for_transfer(X):
#     X_tf = tf.convert_to_tensor(X, dtype=tf.float32)
#     X_tf = tf.image.grayscale_to_rgb(X_tf)
#     X_tf = tf.image.resize(X_tf, TARGET_SIZE)
#     return X_tf
#
# base_model = keras.applications.MobileNetV2(
#     include_top=False,
#     weights="imagenet",
#     input_shape=(*TARGET_SIZE, 3)
# )
# base_model.trainable = False
#
# inputs = keras.Input(shape=(*TARGET_SIZE, 3))
# x = keras.applications.mobilenet_v2.preprocess_input(inputs * 255.0)
# x = base_model(x, training=False)
# x = layers.GlobalAveragePooling2D()(x)
# x = layers.Dropout(0.3)(x)
# outputs = layers.Dense(N_CLASSES, activation="softmax")(x)
#
# transfer_model = keras.Model(inputs, outputs)

### 7.1 Synthèse de l’enrichissement

**TODO :**
- notion ajoutée ;
- raison du choix ;
- modification du pipeline ;
- résultat obtenu ;
- avantage / limite ;
- comparaison au CNN de référence.

## 8. Extension optionnelle — détection et analyse de plusieurs visages

Pipeline attendu :

`Image → détection des visages → bounding boxes → extraction des visages → prétraitement FER2013 → CNN → expression + score`

**Point critique :** le visage extrait doit subir exactement le même prétraitement que pendant l’entraînement :
- taille ;
- grayscale / couleur ;
- normalisation.

### TODO extension
- choisir un détecteur de visages ;
- justifier ce choix ;
- dessiner les bounding boxes ;
- recadrer chaque visage ;
- appliquer le classifieur FER2013 ;
- afficher expression et score.

In [ ]:
# Pseudocode / squelette :
#
# image = ...
# detections = face_detector(image)
#
# for box in detections:
#     face = crop(image, box)
#     face = resize(face, (48, 48))
#     face = to_grayscale(face)
#     face = face.astype("float32") / 255.0
#     face = face[None, ..., None]  # (1, 48, 48, 1)
#
#     proba = final_model.predict(face, verbose=0)[0]
#     pred = np.argmax(proba)
#
#     # dessiner box + CLASS_NAMES[pred] + proba[pred]

## 9. Extension optionnelle — vidéo

Pipeline :

`Vidéo → frames → détection → extraction des visages → CNN → prédictions → affichage`

### Points à surveiller
- coût de calcul ;
- stabilité des prédictions entre frames ;
- fréquence d’inférence ;
- affichage temps réel ;
- cohérence du prétraitement.

# Conclusion générale

**TODO — synthèse finale :**
1. meilleur modèle retenu ;
2. performance validation puis test ;
3. classes les mieux reconnues ;
4. classes les plus difficiles ;
5. principales confusions ;
6. effet des trois expériences ;
7. limites de FER2013 et du modèle ;
8. amélioration prioritaire si le projet devait continuer.

# Checklist avant soutenance

- [ ] Notebook exécutable de bout en bout
- [ ] Source, auteurs et licence du dataset documentés
- [ ] Nombre total d’images + images par classe
- [ ] Exemples visuels par classe
- [ ] Prétraitement expliqué
- [ ] Train / validation / test clairement séparés
- [ ] Baseline dense entraîné
- [ ] CNN personnel construit et justifié
- [ ] Dimensions et paramètres couche par couche
- [ ] Courbes loss / accuracy train + validation
- [ ] Surapprentissage analysé
- [ ] Matrice de confusion
- [ ] Métriques par classe
- [ ] Bonnes et mauvaises prédictions illustrées
- [ ] Au moins 3 expériences comparées
- [ ] Modèle final choisi à partir de la validation
- [ ] Test utilisé seulement à la fin
- [ ] Enrichissement présenté
- [ ] Modèle final sauvegardé
- [ ] Démonstration préparée

## Sauvegarde finale

In [ ]:
# À exécuter lorsque le modèle final est arrêté :
# final_model.save("fer2013_final_model.keras")

# Facultatif : sauvegarde des noms de classes
# pd.Series(CLASS_NAMES).to_json("class_names.json")